In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from pyvis.network import Network
import os
import matplotlib.colors as mcolors

In [2]:
# this folder path needs to be changed for each computer
folder_path='/Users/pipe/Desktop/temp_network_files'
os.chdir(folder_path)

In [18]:
# Loading the graphs
petite_net=nx.read_graphml('petite_network_sim1_step58_n100.graphml')
grande_net=nx.read_graphml('grande_network_sim1_step52_n100.graphml')


In [33]:
# Visualization 1: Motifs of nodes with 2 or more degree-1 neighbors
def create_motif_visualization(graph, filename):
    nt = Network('1000px', '1000px', notebook=True)
    nt.from_nx(graph)
    
    # Identify nodes with 2 or more degree-1 neighbors
    nodes_with_degree1_neighbors = [node for node in graph.nodes() if sum(1 for neighbor in graph.neighbors(node) if graph.degree(neighbor) == 1) >= 2]
    
    for node in nt.nodes:
        node_id = node['id']
        node['size'] = 38
        node['label'] = ''
        
        # Color nodes: red for degree-1 nodes connected to motif centers, red for motif centers, blue for others
        if graph.degree(node_id) == 1 and any(neighbor in nodes_with_degree1_neighbors for neighbor in graph.neighbors(node_id)):
            node['color'] = 'orange'
        elif node_id in nodes_with_degree1_neighbors:
            node['color'] = 'red'
        else:
            node['color'] = 'darkgray'
    
    # Color edges: red for motif edges, blue for others
    for edge in nt.edges:
        if (graph.degree(edge['from']) == 1 and edge['to'] in nodes_with_degree1_neighbors) or \
           (graph.degree(edge['to']) == 1 and edge['from'] in nodes_with_degree1_neighbors):
            edge['color'] = 'red'
            edge['width'] = 18
        else:
            edge['color'] = 'darkgray'
            edge['width'] = 14
    
    nt.toggle_hide_edges_on_drag(True)
    nt.show_buttons(filter_=['physics'])
    nt.set_edge_smooth('dynamic')
    nt.save_graph(filename)

In [34]:
create_motif_visualization(petite_net, 'petite_motif_visualization.html')
create_motif_visualization(grande_net, 'grande_motif_visualization.html')

In [35]:
# Visualization 2: Network diameter edges and nodes highlighted
def create_diameter_visualization(graph, filename):
    nt = Network('1000px', '1000px', notebook=True)
    nt.from_nx(graph)
    
    # Find the diameter path
    def find_diameter_path(graph):
        paths = dict(nx.all_pairs_shortest_path(graph))
        diameter_path = max((path for paths_from_node in paths.values() for path in paths_from_node.values()), key=len)
        return diameter_path
    
    diameter_path = find_diameter_path(graph)
    diameter_edges = list(zip(diameter_path[:-1], diameter_path[1:]))
    diameter_nodes = set(diameter_path)  # Convert to set for faster lookup
    
    # Set all nodes to blue with standard size, then highlight diameter path nodes
    for node in nt.nodes:
        node['label'] = ''
        if node['id'] in diameter_nodes:
            # Diameter path nodes
            node['color'] = 'red'
            node['size'] = 40  # Larger size for diameter path nodes
        else:
            # Regular nodes
            node['color'] = 'darkgray'
            node['size'] = 28
    
    # Color edges: black for diameter path
    for edge in nt.edges:
        if (edge['from'], edge['to']) in diameter_edges or (edge['to'], edge['from']) in diameter_edges:
            edge['color'] = 'red'
            edge['width'] = 25
        else:
            edge['color'] = 'darkgray'
            edge['width'] = 18
    
    nt.toggle_hide_edges_on_drag(True)
    nt.show_buttons(filter_=['physics'])
    nt.set_edge_smooth('dynamic')
    nt.save_graph(filename)

In [36]:
create_diameter_visualization(nx.read_graphml('petite_network_sim1_step74_n150.graphml'), 'petite_diameter_visualization.html')
create_diameter_visualization(nx.read_graphml('grande_network_sim1_step62_n150.graphml'), 'grande_diameter_visualization.html')

In [37]:

# Visualization 3: Highest degree edge highlighted

def calculate_max_edge_degree(network):
    # Initialize result variables
    max_edge_degree = -1
    max_edge = None
    for edge in network.edges():
        node1, node2 = edge
        degree_node1 = network.degree(node1)
        degree_node2 = network.degree(node2)
        edge_degree = degree_node1 + degree_node2 - 2
        if (edge_degree > max_edge_degree):
            max_edge_degree = edge_degree
            max_edge = edge
    return([max_edge, max_edge_degree])

def create_max_edge_degree_visualization(graph, filename):
    nt = Network('1000px', '1000px', notebook=True)
    nt.from_nx(graph)
    
    # Get the edge with maximum degree
    max_edge, max_edge_degree = calculate_max_edge_degree(graph)
    max_edge_nodes = set(max_edge)
    
    # Get neighbors of the max edge nodes
    neighbors_of_max_edge = set()
    for node in max_edge_nodes:
        neighbors_of_max_edge.update(graph.neighbors(node))
    
    # Remove the max edge nodes themselves from neighbors
    neighbors_of_max_edge = neighbors_of_max_edge - max_edge_nodes
    
    # Color nodes
    for node in nt.nodes:
        node_id = node['id']
        node['size'] = 38
        node['label'] = ''
        
        if node_id in max_edge_nodes:
            node['color'] = 'red'  # Max edge nodes in red
        elif node_id in neighbors_of_max_edge:
            node['color'] = 'orange'  # Neighbors in orange
        else:
            node['color'] = 'darkgray'
    
    # Color edges
    for edge in nt.edges:
        edge_tuple = (edge['from'], edge['to'])
        edge_tuple_reverse = (edge['to'], edge['from'])
        
        if edge_tuple == max_edge or edge_tuple_reverse == max_edge:
            edge['color'] = 'red'  # Max degree edge in red
            edge['width'] = 24
        elif (edge['from'] in max_edge_nodes and edge['to'] in neighbors_of_max_edge) or \
             (edge['to'] in max_edge_nodes and edge['from'] in neighbors_of_max_edge):
            edge['color'] = 'orange'  # Edges connecting max nodes to their neighbors
            edge['width'] = 22
        else:
            edge['color'] = 'darkgray'
            edge['width'] = 18
    
    nt.toggle_hide_edges_on_drag(True)
    nt.show_buttons(filter_=['physics'])
    nt.set_edge_smooth('dynamic')
    nt.save_graph(filename)

In [38]:
create_max_edge_degree_visualization(nx.read_graphml('petite_network_sim1_step74_n150.graphml'), 'petite_max_edge_visualization.html')
create_max_edge_degree_visualization(nx.read_graphml('grande_network_sim1_step62_n150.graphml'), 'grande_max_edge_visualization.html')